Defining Envrionment 

In [0]:
# Standard Python utility used to generate a unique ingestion batch ID.
from datetime import datetime, timezone

# Spark SQL helper functions used for dataframe transformations.
from pyspark.sql import functions as F

# Spark data types used later to define a stable schema.
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
)


# A stable project identifier for logs, audit records and documentation.
PROJECT_NAME = "trichy_east_election_pipeline"


# Unity Catalog catalog containing the project tables.
CATALOG = "workspace"

# Database/schema where unprocessed Bronze records will be stored.
BRONZE_SCHEMA = "bronze"


# Root location containing the 2016, 2021 and 2026 source folders.
# Keeping this in one variable makes the pipeline portable.
BASE_VOLUME_PATH = "/Volumes/workspace/elections/trichy_east"


# Assembly constituency identifiers added to every ingested record.
# They provide stable metadata even when a PDF header is unreadable.
CONSTITUENCY_NO = "141"
CONSTITUENCY_NAME = "Tiruchirappalli East"


# Unique identifier for this notebook run.
# It allows every source record to be traced to a specific ingestion run.
BATCH_ID = datetime.now(timezone.utc).strftime(
    "trichy_east_%Y%m%dT%H%M%SZ"
)


# Fully qualified Delta table used to record one audit row per source file.
AUDIT_TABLE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.trichy_east_ingestion_audit"
)


# Display the active configuration before continuing.
# This provides a simple visual check that the notebook targets the right paths.
print("Project:", PROJECT_NAME)
print("Base Volume:", BASE_VOLUME_PATH)
print("Batch ID:", BATCH_ID)
print("Audit table:", AUDIT_TABLE)

In [0]:
# Standard Python library used to read the JSON configuration file.
import json


# The notebook is stored inside the notebooks folder.
# "../config" moves up to the repository root and enters config.
MANIFEST_PATH = "/Workspace/Users/justindhinakar.moorthy@gmail.com/Trichy_East_Election_Data_Pipeline/config/source_manifest.json"


# Load the source manifest maintained in Git.
# Keeping file details outside the notebook makes the pipeline configuration-driven.
with open(MANIFEST_PATH, "r") as manifest_file:
    source_manifest = json.load(manifest_file)


# Read the base Volume path from the manifest.
manifest_base_volume = source_manifest["databricks_base_volume"]


# Check that the notebook and Git configuration point to the same Volume.
# A mismatch could cause files to be read from the wrong location.
assert manifest_base_volume == BASE_VOLUME_PATH, (
    f"Volume path mismatch: notebook={BASE_VOLUME_PATH}, "
    f"manifest={manifest_base_volume}"
)


# Create one configuration record for each expected source file.
source_config_rows = []

for source in source_manifest["sources"]:

    # Combine the base Volume directory with the year-specific relative path.
    source_file_path = (
        f"{BASE_VOLUME_PATH.rstrip('/')}/"
        f"{source['relative_path'].lstrip('/')}"
    )

    source_config_rows.append(
        {
            "project_name": PROJECT_NAME,
            "constituency_no": CONSTITUENCY_NO,
            "constituency_name": CONSTITUENCY_NAME,
            "election_year": int(source["election_year"]),
            "source_type": source["source_type"],
            "source_format": source["format"],
            "source_file_name": source["file_name"],
            "source_file_path": source_file_path,
            "configured_extraction_strategy": source["extraction_strategy"],
            "batch_id": BATCH_ID,
        }
    )


# Convert the Python records into a Spark dataframe.
# This dataframe becomes the control list for the ingestion pipeline.
source_config_df = spark.createDataFrame(source_config_rows)


# Show all expected files in a predictable order.
display(
    source_config_df.orderBy(
        "election_year",
        "source_type"
    )
)


print("Configured source files:", source_config_df.count())

In [0]:
# Python utilities used to inspect files mounted under /Volumes.
import os
from pathlib import Path


# Store one validation result for each configured source file.
file_check_rows = []

for source in source_config_rows:
    source_path = source["source_file_path"]

    try:
        # Check whether the configured file is accessible.
        file_exists = os.path.isfile(source_path)

        if file_exists:
            # Capture basic file metadata for auditing.
            file_stats = os.stat(source_path)

            file_size_bytes = int(file_stats.st_size)
            file_modified_timestamp = datetime.fromtimestamp(
                file_stats.st_mtime,
                tz=timezone.utc
            )
            file_status = "AVAILABLE"
            validation_error = None
        else:
            file_size_bytes = None
            file_modified_timestamp = None
            file_status = "MISSING"
            validation_error = "Configured source file was not found."

    except Exception as error:
        # Preserve unexpected access errors instead of silently skipping files.
        file_exists = False
        file_size_bytes = None
        file_modified_timestamp = None
        file_status = "ERROR"
        validation_error = str(error)

    file_check_rows.append(
        {
            **source,
            "file_extension": Path(source_path).suffix.lower(),
            "file_exists": file_exists,
            "file_size_bytes": file_size_bytes,
            "file_modified_timestamp": file_modified_timestamp,
            "file_status": file_status,
            "validation_error": validation_error,
        }
    )


# Convert the validation results into a Spark dataframe.
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    LongType,
    BooleanType,
    TimestampType,
)


# Define types explicitly so Spark can handle columns containing only nulls.
source_inventory_schema = StructType([
    StructField("project_name", StringType(), False),
    StructField("constituency_no", StringType(), False),
    StructField("constituency_name", StringType(), False),
    StructField("election_year", IntegerType(), False),
    StructField("source_type", StringType(), False),
    StructField("source_format", StringType(), False),
    StructField("source_file_name", StringType(), False),
    StructField("source_file_path", StringType(), False),
    StructField("configured_extraction_strategy", StringType(), False),
    StructField("batch_id", StringType(), False),
    StructField("file_extension", StringType(), True),
    StructField("file_exists", BooleanType(), False),
    StructField("file_size_bytes", LongType(), True),
    StructField("file_modified_timestamp", TimestampType(), True),
    StructField("file_status", StringType(), False),
    StructField("validation_error", StringType(), True),
])


source_inventory_df = spark.createDataFrame(
    file_check_rows,
    schema=source_inventory_schema
)


# Display the result for every expected source.
display(
    source_inventory_df.select(
        "election_year",
        "source_type",
        "source_file_name",
        "source_file_path",
        "file_size_bytes",
        "file_status",
        "validation_error"
    ).orderBy(
        "election_year",
        "source_type"
    )
)


# Stop the notebook when configured files are missing.
unavailable_file_count = (
    source_inventory_df
    .filter(F.col("file_status") != "AVAILABLE")
    .count()
)

print("Configured files:", source_inventory_df.count())
print("Unavailable files:", unavailable_file_count)

if unavailable_file_count > 0:
    raise ValueError(
        f"{unavailable_file_count} configured source file(s) are unavailable."
    )

In [0]:
# Python library used to generate SHA-256 file checksums.
import hashlib


def calculate_sha256(file_path, chunk_size=1024 * 1024):
    """
    Calculate a SHA-256 checksum without loading the entire file into memory.

    Reading in chunks is safer for large PDFs and keeps driver memory usage low.
    """
    checksum = hashlib.sha256()

    with open(file_path, "rb") as source_file:
        while True:
            chunk = source_file.read(chunk_size)

            if not chunk:
                break

            checksum.update(chunk)

    return checksum.hexdigest()


# Calculate one checksum for every available source file.
checksum_rows = []

for source in file_check_rows:
    checksum_error = None
    source_checksum = None

    if source["file_status"] == "AVAILABLE":
        try:
            source_checksum = calculate_sha256(
                source["source_file_path"]
            )
        except Exception as error:
            checksum_error = str(error)

    checksum_rows.append(
        {
            **source,
            "source_sha256": source_checksum,
            "checksum_error": checksum_error,
        }
    )


# Explicit types avoid inference errors when every error value is null.
checksum_schema = (
    source_inventory_schema
    .add(StructField("source_sha256", StringType(), True))
    .add(StructField("checksum_error", StringType(), True))
)


source_inventory_with_checksum_df = spark.createDataFrame(
    checksum_rows,
    schema=checksum_schema
)


# Display shortened checksums for readability.
display(
    source_inventory_with_checksum_df
    .select(
        "election_year",
        "source_type",
        "source_file_name",
        "file_size_bytes",
        F.substring("source_sha256", 1, 12).alias("sha256_prefix"),
        "checksum_error"
    )
    .orderBy("election_year", "source_type")
)


checksum_failure_count = (
    source_inventory_with_checksum_df
    .filter(
        F.col("source_sha256").isNull()
        | F.col("checksum_error").isNotNull()
    )
    .count()
)

print("Checksum failures:", checksum_failure_count)

if checksum_failure_count > 0:
    raise ValueError(
        f"Checksum generation failed for {checksum_failure_count} source file(s)."
    )

In [0]:
# Create the Bronze schema if it does not already exist.
# IF NOT EXISTS makes this safe to rerun.
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}
""")


# Create the audit table using an explicit schema.
# The table stores one record per source file per ingestion batch.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {AUDIT_TABLE} (
    project_name STRING,
    constituency_no STRING,
    constituency_name STRING,
    election_year INT,

    source_type STRING,
    source_format STRING,
    source_file_name STRING,
    source_file_path STRING,

    configured_extraction_strategy STRING,

    file_extension STRING,
    file_exists BOOLEAN,
    file_size_bytes BIGINT,
    file_modified_timestamp TIMESTAMP,
    file_status STRING,

    source_sha256 STRING,

    validation_error STRING,
    checksum_error STRING,

    batch_id STRING,
    audit_status STRING,
    audit_timestamp TIMESTAMP
)
USING DELTA
""")


print("Bronze schema:", f"{CATALOG}.{BRONZE_SCHEMA}")
print("Audit table created:", AUDIT_TABLE)

In [0]:
# Convert file validation results into the audit table structure.
audit_batch_df = (
    source_inventory_with_checksum_df
    .withColumn(
        "audit_status",
        F.when(
            (F.col("file_status") == "AVAILABLE")
            & F.col("source_sha256").isNotNull()
            & F.col("validation_error").isNull()
            & F.col("checksum_error").isNull(),
            F.lit("PASS")
        ).otherwise(F.lit("FAIL"))
    )
    .withColumn(
        "audit_timestamp",
        F.current_timestamp()
    )
    .select(
        "project_name",
        "constituency_no",
        "constituency_name",
        "election_year",
        "source_type",
        "source_format",
        "source_file_name",
        "source_file_path",
        "configured_extraction_strategy",
        "file_extension",
        "file_exists",
        "file_size_bytes",
        "file_modified_timestamp",
        "file_status",
        "source_sha256",
        "validation_error",
        "checksum_error",
        "batch_id",
        "audit_status",
        "audit_timestamp"
    )
)


# Remove the current batch before writing it.
# This makes rerunning this cell idempotent for the same BATCH_ID.
spark.sql(f"""
DELETE FROM {AUDIT_TABLE}
WHERE batch_id = '{BATCH_ID}'
""")


# Append the current batch to the Delta audit table.
(
    audit_batch_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(AUDIT_TABLE)
)


print("Audit rows written:", audit_batch_df.count())
print("Batch ID:", BATCH_ID)
print("Audit table:", AUDIT_TABLE)